In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('Bank_Churn_messy.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

In [ ]:
print(df.duplicated().sum())
print(df['CustomerId'].duplicated().sum())

df = df.drop_duplicates()
df = df.drop_duplicates(subset='CustomerId', keep='first')
print(df.shape)

In [ ]:
df['Surname'] = df['Surname'].str.strip().str.title()

geo_map = {'fr': 'France', 'france': 'France', 'esp': 'Spain', 'spain': 'Spain', 'germany': 'Germany'}
df['Geography'] = df['Geography'].str.strip().str.lower().map(geo_map)

df['Gender'] = (df['Gender'].str.strip().str.lower()
                .replace({'f': 'female', 'm': 'male'})
                .str.title())

print(df['Geography'].value_counts(dropna=False))
print(df['Gender'].value_counts(dropna=False))

In [ ]:
limits = {'CreditScore': (350, 850), 'Age': (18, 92), 'Tenure': (0, 10)}

for col, (low, high) in limits.items():
    bad = (df[col] < low) | (df[col] > high)
    print(col, int(bad.sum()))
    df.loc[bad, col] = np.nan

neg_salary = df['EstimatedSalary'] < 0
print('EstimatedSalary', int(neg_salary.sum()))
df.loc[neg_salary, 'EstimatedSalary'] = np.nan

In [ ]:
df['BalanceMissing'] = df['Balance'].isna().astype(int)
print(df.groupby('BalanceMissing')['Exited'].mean())

df['Balance'] = df['Balance'].fillna(0.0)

for col in ['CreditScore', 'Age', 'Tenure', 'EstimatedSalary']:
    df[col] = df[col].fillna(df[col].median())

df['Geography'] = df['Geography'].fillna(df['Geography'].mode()[0])

df['Age'] = df['Age'].astype(int)
df['Tenure'] = df['Tenure'].astype(int)

print(df.isna().sum())
print(df.shape)

In [ ]:
churn_counts = df['Exited'].value_counts().sort_index()
churn_rate = df['Exited'].mean()

print(churn_counts)
print('churn rate:', round(churn_rate * 100, 2), '%')
print('majority class baseline accuracy:', round((1 - churn_rate) * 100, 2), '%')

ax = churn_counts.plot(kind='bar', color=['#4c72b0', '#c44e52'], figsize=(5, 4))
ax.set_xticklabels(['Retained (0)', 'Churned (1)'], rotation=0)
ax.set_ylabel('Customers')
ax.set_title(f'Class balance — churn rate {churn_rate:.1%}')
plt.tight_layout()
plt.show()

In [ ]:
gender_churn = df.groupby('Gender')['Exited'].agg(customers='count', churned='sum', churn_rate='mean')
gender_churn['churn_rate_pct'] = (gender_churn['churn_rate'] * 100).round(2)
print(gender_churn)

chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(df['Gender'], df['Exited']))
print('chi2 =', round(chi2, 2), '| p =', f'{p:.3e}')
print('gap (pp):', round((gender_churn['churn_rate'].max() - gender_churn['churn_rate'].min()) * 100, 2))

sns.barplot(data=df, x='Gender', y='Exited', errorbar=('ci', 95))
plt.axhline(churn_rate, ls='--', c='grey', label='overall')
plt.ylabel('Churn rate')
plt.legend()
plt.title('Churn rate by Gender')
plt.tight_layout()
plt.show()

In [ ]:
card_churn = df.groupby('HasCrCard')['Exited'].agg(customers='count', churned='sum', churn_rate='mean')
card_churn['churn_rate_pct'] = (card_churn['churn_rate'] * 100).round(2)
print(card_churn)

chi2, p, dof, _ = stats.chi2_contingency(pd.crosstab(df['HasCrCard'], df['Exited']))
print('chi2 =', round(chi2, 3), '| p =', round(p, 4))
print('gap (pp):', round((card_churn['churn_rate'].max() - card_churn['churn_rate'].min()) * 100, 2))
print('point-biserial r:', round(stats.pointbiserialr(df['HasCrCard'], df['Exited'])[0], 4))

In [ ]:
tenure_churn = df.groupby('Tenure')['Exited'].agg(customers='count', churned='sum', churn_rate='mean')
tenure_churn['churn_rate_pct'] = (tenure_churn['churn_rate'] * 100).round(2)
print(tenure_churn)

r, p = stats.pearsonr(df['Tenure'], df['Exited'])
print('pearson r =', round(r, 4), '| p =', round(p, 4))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tenure_churn.index, tenure_churn['churn_rate'], marker='o', color='#c44e52')
ax.axhline(churn_rate, ls='--', c='grey', label=f'overall {churn_rate:.1%}')
ax.set_xlabel('Tenure (years with bank)')
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate vs Tenure')
ax.set_xticks(tenure_churn.index)
ax.legend()
plt.tight_layout()
plt.savefig('churn_rate_by_tenure.png', dpi=150)
plt.show()

In [ ]:
print('zero-balance share:', round((df['Balance'] == 0).mean() * 100, 2), '%')
print(df.groupby(df['Balance'] == 0)['Exited'].agg(customers='count', churn_rate='mean'))

df['BalanceQuartile'] = pd.qcut(df['Balance'], 4, duplicates='drop')
bal_churn = df.groupby('BalanceQuartile', observed=True)['Exited'].agg(customers='count', churn_rate='mean')
bal_churn['churn_rate_pct'] = (bal_churn['churn_rate'] * 100).round(2)
print(bal_churn)

funded = df[df['Balance'] > 0].copy()
funded['Quartile'] = pd.qcut(funded['Balance'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(funded.groupby('Quartile', observed=True)['Exited'].agg(customers='count', churn_rate='mean'))

sns.barplot(x=bal_churn.index.astype(str), y=bal_churn['churn_rate'], color='#4c72b0')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Churn rate')
plt.title('Churn rate by Balance bucket')
plt.tight_layout()
plt.show()

In [ ]:
num = df.select_dtypes(include=np.number).drop(columns=['CustomerId', 'BalanceMissing'])
corr_with_target = num.corr()['Exited'].drop('Exited').sort_values(key=abs, ascending=False)
print(corr_with_target.round(4))

df['IsFemale'] = df['Gender'].eq('Female').astype(int)
df['IsGermany'] = df['Geography'].eq('Germany').astype(int)
print('IsFemale  ', round(df['IsFemale'].corr(df['Exited']), 4))
print('IsGermany ', round(df['IsGermany'].corr(df['Exited']), 4))

plt.figure(figsize=(8, 6))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7})
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
print(df.groupby('Geography')['Exited'].agg(customers='count', churn_rate='mean'))
print(df.groupby('IsActiveMember')['Exited'].agg(customers='count', churn_rate='mean'))
print(df.groupby('NumOfProducts')['Exited'].agg(customers='count', churn_rate='mean'))

bins = [17, 30, 40, 50, 60, 95]
df['AgeBand'] = pd.cut(df['Age'], bins=bins)
print(df.groupby('AgeBand', observed=True)['Exited'].agg(customers='count', churn_rate='mean'))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.barplot(data=df, x='Geography', y='Exited', ax=axes[0], errorbar=None)
sns.barplot(data=df, x='NumOfProducts', y='Exited', ax=axes[1], errorbar=None)
sns.barplot(data=df, x='AgeBand', y='Exited', ax=axes[2], errorbar=None)
for a in axes:
    a.axhline(churn_rate, ls='--', c='grey')
    a.set_ylabel('Churn rate')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    'metric': ['rows after cleaning', 'overall churn rate', 'female churn', 'male churn',
               'has card churn', 'no card churn', 'strongest numeric corr', 'zero-balance churn',
               'funded-balance churn', 'Germany churn', 'inactive churn', '3+ products churn'],
    'value': [len(df),
              f"{df['Exited'].mean():.2%}",
              f"{df.loc[df.Gender == 'Female', 'Exited'].mean():.2%}",
              f"{df.loc[df.Gender == 'Male', 'Exited'].mean():.2%}",
              f"{df.loc[df.HasCrCard == 1, 'Exited'].mean():.2%}",
              f"{df.loc[df.HasCrCard == 0, 'Exited'].mean():.2%}",
              f"Age (r={num.corr()['Exited']['Age']:.3f})",
              f"{df.loc[df.Balance == 0, 'Exited'].mean():.2%}",
              f"{df.loc[df.Balance > 0, 'Exited'].mean():.2%}",
              f"{df.loc[df.Geography == 'Germany', 'Exited'].mean():.2%}",
              f"{df.loc[df.IsActiveMember == 0, 'Exited'].mean():.2%}",
              f"{df.loc[df.NumOfProducts >= 3, 'Exited'].mean():.2%}"]
})
print(summary.to_string(index=False))

df.to_csv('Bank_Churn_clean.csv', index=False)